# A model-driven agent on AgentDojo (Colab T4)

Pre-registered in `research/experiments/model_agent/README.md`. Read it before running:
it fixes the arms, the hypotheses and the gates, and this notebook does not change them.

**What this notebook is.** The first Tekmor run where the agent is a *model* rather than
AgentDojo's ground truth. Every AgentDojo number in `docs/decisions.md` was produced by a
driver that replays the oracle trace and obeys every injection, which makes ASR an
always-obeys bound, makes BTU a question about the policy rather than about work getting
done, and makes provenance near-oracle. This removes that confound.

**What this notebook is not.** It is **not the full pre-registered run**. A T4 cannot
generate for 680 runs x 6 arms in a session — the cost cell below makes you measure that
before you commit to anything. What fits here is a *pilot*: a subsample that establishes
the loop works, measures what a full run would cost, and gives directional answers to H3
and H4. A subsampled result is not the pre-registered result and must not be recorded as
one.

**Run the cells in order.** `evaluation/results/` is gitignored and this VM is
ephemeral, so the last two cells print everything into the notebook output and offer a
download. If you skip them, the run is lost.

## 1. The GPU

Expect a Tesla T4, 16 GB. Turing has no bfloat16, so everything below is float16.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone and install

The repository is private, so this asks for a GitHub token. In Colab the prompt appears
under the cell; in VS Code it appears in the input box at the top of the window. The
token travels in the environment and never in `argv`, because a failed clone prints
`argv` into the traceback and the traceback into the saved notebook.

In [ ]:
import base64
import getpass
import os
import subprocess
import sys

BRANCH = "feat/model-driven-agent"
REPO = "https://github.com/ghassenov/Tekmor.git"


def git(*args, env=None):
    """Run git and raise with git's own message: a CalledProcessError alone says nothing."""
    done = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{done.stderr.strip() or done.stdout.strip()}")
    return done.stdout


token = getpass.getpass("GitHub token: ").strip()
if not token:
    raise RuntimeError("empty token: rerun and paste it into the prompt")
basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = os.environ | {
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.extraHeader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}",
}
# Check the branch is on the remote before cloning. "not found in upstream origin"
# buried in a clone traceback is the one failure here that wastes the most time,
# and a branch that exists only on someone's laptop is the usual cause.
if BRANCH not in git("ls-remote", "--heads", REPO, BRANCH, env=env):
    raise RuntimeError(
        f"branch {BRANCH!r} is not on {REPO}.\n"
        "Push it, or set BRANCH above to one that is (`git ls-remote --heads <repo>`)."
    )
if os.path.isdir("/content/Tekmor/.git"):
    # Update in place. `git pull` cannot work here: the credential lived in one
    # subprocess environment and was never stored, so this checkout has no auth.
    git("-C", "/content/Tekmor", "fetch", "-q", "origin", BRANCH, env=env)
    git("-C", "/content/Tekmor", "reset", "-q", "--hard", "FETCH_HEAD", env=env)
else:
    git("clone", "-q", "--branch", BRANCH, REPO, "/content/Tekmor", env=env)
del token, basic, env
%cd /content/Tekmor
# Which commit this run is against: every result below has to name it.
!git log --oneline -n 1 && git rev-parse --abbrev-ref HEAD
# Tekmor has no runtime dependencies, so src/ on the path is the whole install.
sys.path.insert(0, "/content/Tekmor/src")
# agentdojo is the benchmark; bitsandbytes is the 4-bit loader.
!pip install -q "agentdojo>=0.1.35" bitsandbytes accelerate

In [ ]:
# Fail now, loudly, rather than after an hour of generation.
import importlib.metadata as md

import torch

from evaluation.dojo import VERSION

print("agentdojo :", md.version("agentdojo"), "| suites pinned at", VERSION)
print("torch     :", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU"

## 3. Configure the run

`AGENT_MODEL` is the reference model this project has used on a T4 (`docs/decisions.md`
records Qwen3-8B NF4 as the judge). Quantised weights are a *different model* from the
full-precision one, and the manifest records that.

`LIMIT` keeps the first N user tasks and N injection tasks per suite, so the cost grows
roughly as `N + N*N` runs per suite per arm. Start small. The next cell measures what
your choice actually costs before you spend it.

In [ ]:
AGENT_MODEL = "Qwen/Qwen3-8B"
AGENT_QUANT = "nf4"  # 8B does not fit on a 16 GB T4 unquantised
AGENT_DTYPE = "float16"  # Turing has no bfloat16
MAX_ITERS = 15  # AgentDojo's own default tool-loop depth
MAX_NEW_TOKENS = 512  # a truncated turn loses its tool call and ends the run

# One suite to calibrate. The full set is "banking slack travel workspace".
SUITES = "banking"
LIMIT = 2  # user tasks and injection tasks kept per suite

# hf-native: the model's own tool-call format. The earlier "hf" (AgentDojo's
# <function=...> convention) made Qwen3 narrate instead of calling tools after the first
# turn, which scored every arm at zero. "hf" stays available to measure that difference.
AGENT = (
    f"--agent hf-native --agent-model {AGENT_MODEL} --agent-quant {AGENT_QUANT} "
    f"--agent-dtype {AGENT_DTYPE} --agent-max-iters {MAX_ITERS} "
    f"--agent-max-new-tokens {MAX_NEW_TOKENS}"
)
print(AGENT)

## 4. Smoke test: does the loop work on this GPU?

One suite, one task, a shallow loop. This is not a measurement — it is a check that the
model loads, generates, produces parseable tool calls, and that refusals come back to it.
Weights download on first use (~6 GB), so this cell is slow once and fast after.

**Read the `inv` column.** It counts pairs whose attack does not land even under
`allow-all`. With a truncated loop that will be most of them, which is correct and is why
this is not a result.

In [ ]:
!PYTHONPATH=src python -m evaluation.dojo --suites banking --limit 1 $AGENT

## 4a. Stop here if the agent cannot do the work

**This cell stops the notebook if `allow-all` finished no benign task.**

`allow-all` is the run with no defense at all. If it scores zero, the agent could not do
the user's job unaided, and every arm below then compares zero against zero: a defense
cannot cost utility that was never there. Running on would spend hours of GPU on numbers
that cannot answer anything.

The first pilot of this notebook did exactly that, for two hours, because this check was
a note at the bottom instead of a stop.

In [ ]:
import json
import pathlib

newest = max(
    pathlib.Path("evaluation/results/processed").glob("*-agentdojo*"),
    key=lambda p: p.stat().st_mtime,
)
rows = json.loads((newest / "metrics.json").read_text())
ceiling = next(r for r in rows if r["defense"] == "allow-all")
print(newest.name)
print(f"allow-all BTU = {ceiling['btu']:.2f} over {ceiling['benign_runs']} benign run(s)")

if ceiling["btu"] == 0.0:
    raise SystemExit(
        "STOP: the undefended agent finished no benign task, so there is no utility "
        "ceiling and nothing below can be interpreted. Run section 4b to see what the "
        "model emits, raise MAX_NEW_TOKENS, or use a model that can drive these tools."
    )
print("ceiling is non-zero: the arms below have something to be measured against")

## 4b. Diagnostic: watch one whole run, turn by turn

Runs a single benign task under **`allow-all`** — no defense at all — and prints every
turn: what the model wrote, what tool call was parsed out of it, and what the task
actually required. Nothing here is a measurement; it is for finding where the run stops
matching the job.

Three things to compare:

- **GROUND TRUTH** is the call sequence AgentDojo considers a correct solution.
- **NO TOOL CALL** on a turn means the loop ended there. That is how a run finishes
  early without finishing the task.
- **utility** is AgentDojo's own verdict on whether the task got done.

In [ ]:
from agentdojo.attacks.attack_registry import load_attack
from agentdojo.task_suite.load_suites import get_suites

from evaluation import dojo
from tekmor.defense.baselines import AllowAll

suite = get_suites(dojo.VERSION)["banking"]
user_task = list(suite.user_tasks.values())[0]

client = dojo.HFChatClient(AGENT_MODEL, AGENT_QUANT, AGENT_DTYPE, MAX_NEW_TOKENS)
llm = dojo.HFToolCallingLLM(client)

turns = []
inner = client.generate


def tee(messages, tools=None):
    out = inner(messages, tools=tools)
    turns.append(out)
    return out


client.generate = tee

record = dojo.run_pair(
    "banking",
    suite,
    AllowAll(),
    user_task,
    None,
    load_attack("direct", suite, None),
    agent=dojo.model_agent(llm, MAX_ITERS),
)

print("TASK:", user_task.PROMPT)
print("utility (AgentDojo: was the task completed?):", record.utility)
print(f"turns: {len(turns)}, of which {llm.no_tool_call} produced no tool call")

env = user_task.init_environment(suite.load_and_inject_default_environment({}))
print("\nGROUND TRUTH the task expects")
print("=" * 78)
for call in user_task.ground_truth(env):
    print(" ", call.function, dict(call.args))

print("\nWHAT THE MODEL DID")
print("=" * 78)
for i, text in enumerate(turns):
    calls = dojo.HFToolCallingLLM._parse(text)
    shown = [(c.function, dict(c.args)) for c in calls] or "NO TOOL CALL -> loop ends here"
    # Length near MAX_NEW_TOKENS*3 chars means the turn was cut off, which loses a call
    # that was being written and looks identical to a model that chose not to call one.
    print(f"--- turn {i} ({len(text)} chars): {shown}")
    print(repr(text[:700]))

## 5. What a real run would cost

Measures wall-clock for a known number of runs and projects the rest. **Look at the
projection before running anything below it.** If the full pre-registered run does not
fit in a session, the honest move is to shrink the arms or the suites and say so — not to
start it and lose it to a disconnect.

In [ ]:
import time

t0 = time.perf_counter()
!PYTHONPATH=src python -m evaluation.dojo --suites banking --limit 1 $AGENT
elapsed = time.perf_counter() - t0

# --limit 1 on one suite = (1 benign + 1 attacked) x 4 defenses = 8 runs.
per_run = elapsed / 8
full_runs = (97 + 583) * 4  # the whole benchmark, 4 defenses in one invocation
print(f"\n{elapsed / 60:.1f} min for 8 runs -> {per_run:.1f} s/run")
print(f"one full-benchmark invocation ~ {per_run * full_runs / 3600:.1f} h")
arms = per_run * full_runs * 5 / 3600  # roughly: arm 6d runs 3 defenses, not 4
print(f"all five arms below, in full  ~ {arms:.1f} h")
print(
    f"\nyour pilot ({SUITES}, limit {LIMIT}) per invocation ~ "
    f"{per_run * len(SUITES.split()) * (LIMIT + LIMIT * LIMIT) * 4 / 60:.0f} min"
)

## 6. The arms

Each cell is one invocation and writes its own timestamped results directory and
manifest. They are independent: run the ones you can afford, in any order, and record
which ones you ran.

`allow-all` is in every invocation and is not a formality here. Under a model agent it is
the ceiling that separates the model's own competence from the defense's cost, and every
utility claim in the pre-registration is relative to it (H2).

### 6a. Baselines and the core — H1 and H2

In [ ]:
!PYTHONPATH=src python -m evaluation.dojo --suites $SUITES --limit $LIMIT $AGENT

### 6b. With endorsement

In [ ]:
!PYTHONPATH=src python -m evaluation.dojo --suites $SUITES --limit $LIMIT --endorse $AGENT

### 6c. Argument provenance, with and without field labels — H3

H3 predicts field labels now **cost** utility, because a model that reformats a value
makes it untraced, and untraced falls back to call level. The field-label experiment
measured zero cost on the ground-truth driver and said in advance that the zero was an
artifact of verbatim copying. These two cells are that prediction's test, so run both or
neither — one without the other answers nothing.

In [ ]:
!PYTHONPATH=src python -m evaluation.dojo --suites $SUITES --limit $LIMIT --arguments $AGENT

In [ ]:
!PYTHONPATH=src python -m evaluation.dojo --suites $SUITES --limit $LIMIT \
    --arguments --field-labels $AGENT

### 6d. deny-gray — H4

The gray-zone refusal switch, through the alignment driver with no judge (`--judge none`,
so no second model is loaded). On the ground-truth driver this dominated the core: equal
BTU, every remaining attack removed, while refusing roughly half of all benign actions.
H4 predicts a model that actually has to finish the task pays for those refusals.

This also runs Tekmor's own 26-scenario matrix on CPU first, which is quick; the AgentDojo
half is the slow part.

In [ ]:
!PYTHONPATH=src python -m evaluation.alignment --judge none --endorse --dojo \
    --suites $SUITES --limit $LIMIT $AGENT

## 7. Bring the results back

`evaluation/results/` is gitignored and this VM is ephemeral. The Proposal B GPU run lost
its `runs.jsonl` and `decisions.jsonl` exactly this way, and only survived because its
metrics were printed into the notebook output. Do both: print, then download.

In [ ]:
import json
import pathlib

# Everything this session wrote, printed verbatim so it survives in the saved notebook.
for directory in sorted(pathlib.Path("evaluation/results/processed").glob("*")):
    print("=" * 78)
    print(directory.name)
    for f in sorted(directory.glob("*.json")):
        print("-" * 78, f.name, sep="\n")
        print(json.dumps(json.loads(f.read_text()), indent=1, sort_keys=True))
    manifest = pathlib.Path("evaluation/results/raw") / directory.name / "manifest.json"
    if manifest.exists():
        print("-" * 78, "manifest.json", sep="\n")
        print(manifest.read_text())

In [ ]:
# The raw per-run records too, which the printout above does not include.
!cd /content/Tekmor && zip -qr /content/tekmor-model-agent-results.zip evaluation/results
print("zipped:", __import__("os").path.getsize("/content/tekmor-model-agent-results.zip"), "bytes")
try:
    from google.colab import files

    files.download("/content/tekmor-model-agent-results.zip")
except ImportError:
    print("not on Colab: copy /content/tekmor-model-agent-results.zip yourself")

## 8. Before you record anything

- Which arms actually ran, at which `SUITES` and `LIMIT`. A subsample is not the
  pre-registered run, and the entry must say so in its first line.
- `agent_empty_completions` in each manifest. An empty completion parses as "no tool
  calls" and ends a run, which is indistinguishable from an agent that finished. If that
  count is not near zero, the utility numbers are measuring truncation, not defense.
- The `inv` column per arm. Pairs whose attack does not land under `allow-all` are
  excluded from ASR, and a weak agent makes many of them invalid — that is H2 biting, and
  it belongs in the write-up rather than being quietly dropped.
- Commit the results entry to `docs/decisions.md` with the driver named in the table, and
  never in the same table as a ground-truth number.